### Задание 1: Создание user-item-матрицы, разбиение данных на тест и контроль

In [14]:
# Шаг 1.1: Подготовка окружения и загрузка данных
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from pandas.api.types import CategoricalDtype
import implicit
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
ratings = pd.read_csv('ratings.csv')
books = pd.read_csv('books.csv')

# Проверка данных
print("Рейтинги:", ratings.shape)
print("Книги:", books.shape)
print(ratings.head())
print(books.head())

Рейтинги: (5976479, 3)
Книги: (10000, 23)
   user_id  book_id  rating
0        1      258       5
1        2     4081       4
2        2      260       5
3        2     9296       5
4        2     2318       3
   book_id  goodreads_book_id  best_book_id  work_id  books_count       isbn  \
0        1            2767052       2767052  2792775          272  439023483   
1        2                  3             3  4640799          491  439554934   
2        3              41865         41865  3212258          226  316015849   
3        4               2657          2657  3275794          487   61120081   
4        5               4671          4671   245494         1356  743273567   

         isbn13                      authors  original_publication_year  \
0  9.780439e+12              Suzanne Collins                     2008.0   
1  9.780440e+12  J.K. Rowling, Mary GrandPré                     1997.0   
2  9.780316e+12              Stephenie Meyer                     2005.0   
3  9.7800

In [15]:
# Шаг 1.2: Преобразование рейтингов в бинарные оценки
# Преобразуем рейтинги в бинарные: 1 - книга понравилась (оценка 4 или 5), 0 - не понравилась
ratings['rating_binary'] = ratings['rating'].apply(lambda x: 1 if x >= 4 else 0)
print("Распределение бинарных оценок:")
print(ratings['rating_binary'].value_counts())

Распределение бинарных оценок:
rating_binary
1    4122111
0    1854368
Name: count, dtype: int64


In [16]:
# Шаг 1.3: Нумерация прочитанных книг каждого пользователя
# Для каждого пользователя пронумеруем его прочитанные книги в порядке появления
ratings['book_order'] = ratings.groupby('user_id')['book_id'].rank(method='first')
print("Пример нумерации книг для первых пользователей:")
print(ratings[['user_id', 'book_id', 'rating', 'book_order']].head(10))

Пример нумерации книг для первых пользователей:
   user_id  book_id  rating  book_order
0        1      258       5        63.0
1        2     4081       4        54.0
2        2      260       5        30.0
3        2     9296       5        63.0
4        2     2318       3        48.0
5        2       26       4        11.0
6        2      315       3        35.0
7        2       33       4        14.0
8        2      301       5        33.0
9        2     2686       5        49.0


In [17]:
# Шаг 1.4: Расчёт долей прочитанных книг
# Рассчитаем общее количество прочитанных книг для каждого пользователя
total_books_per_user = ratings.groupby('user_id')['book_id'].count()
ratings = ratings.merge(total_books_per_user.rename('total_books'), on='user_id')

# Переведём номера в доли: порядковый номер / общее количество
ratings['book_fraction'] = ratings['book_order'] / ratings['total_books']
print("Пример долей прочитанных книг:")
print(ratings[['user_id', 'book_id', 'book_order', 'total_books', 'book_fraction']].head(10))

Пример долей прочитанных книг:
   user_id  book_id  book_order  total_books  book_fraction
0        1      258        63.0          117       0.538462
1        2     4081        54.0           65       0.830769
2        2      260        30.0           65       0.461538
3        2     9296        63.0           65       0.969231
4        2     2318        48.0           65       0.738462
5        2       26        11.0           65       0.169231
6        2      315        35.0           65       0.538462
7        2       33        14.0           65       0.215385
8        2      301        33.0           65       0.507692
9        2     2686        49.0           65       0.753846


In [18]:
# Шаг 1.5: Разбиение данных на train и control (70/30)
# 70% книг каждого пользователя - на обучение, 30% - на контроль
TRAIN_RATIO = 0.7
ratings['is_train'] = ratings['book_fraction'] <= TRAIN_RATIO

train_ratings = ratings[ratings['is_train']].copy()
test_ratings = ratings[~ratings['is_train']].copy()

print(f"Размер обучающей выборки: {len(train_ratings)}")
print(f"Размер тестовой выборки: {len(test_ratings)}")
print(f"Процент в train: {len(train_ratings) / len(ratings) * 100:.2f}%")
print(f"Процент в test: {len(test_ratings) / len(ratings) * 100:.2f}%")

Размер обучающей выборки: 4159553
Размер тестовой выборки: 1816926
Процент в train: 69.60%
Процент в test: 30.40%


In [19]:
# Шаг 1.6: Создание user-item-матрицы для обучения
# Создаём user-item матрицу для данных обучения
# Получаем уникальных пользователей и книги из train
user_index = train_ratings['user_id'].unique()
book_index = train_ratings['book_id'].unique()

print(f"Уникальных пользователей в train: {len(user_index)}")
print(f"Уникальных книг в train: {len(book_index)}")

# Преобразуем user_id и book_id в индексы матрицы
rows = train_ratings['user_id'].astype(CategoricalDtype(categories=user_index)).cat.codes
cols = train_ratings['book_id'].astype(CategoricalDtype(categories=book_index)).cat.codes

# Создаём разреженную матрицу (sparse)
train_matrix = sparse.csr_matrix(
    (train_ratings['rating_binary'].values, (rows.values, cols.values)), 
    shape=(len(user_index), len(book_index))
)

print(f"\nРазмерность user-item матрицы: {train_matrix.shape}")
print(f"Количество ненулевых элементов: {train_matrix.nnz}")
print(f"Разреженность: {1 - train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1]):.4f}")

Уникальных пользователей в train: 53424
Уникальных книг в train: 6654

Размерность user-item матрицы: (53424, 6654)
Количество ненулевых элементов: 4159553
Разреженность: 0.9883


In [20]:
# Шаг 1.7: Сохранение словарей соответствия индексов
# Сохраняем соответствие индексов для использования в следующих заданиях
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(user_index)}
idx_to_user_id = {idx: user_id for idx, user_id in enumerate(user_index)}

book_id_to_idx = {book_id: idx for idx, book_id in enumerate(book_index)}
idx_to_book_id = {idx: book_id for idx, book_id in enumerate(book_index)}

print("Словари соответствия созданы:")
print(f"user_id_to_idx: {len(user_id_to_idx)} пользователей")
print(f"book_id_to_idx: {len(book_id_to_idx)} книг")

Словари соответствия созданы:
user_id_to_idx: 53424 пользователей
book_id_to_idx: 6654 книг


### Задание 2: Применение метода матричной факторизации и сбор признаков для контентной модели

In [21]:
# Шаг 2.1: Реализация функции AP@K
def average_precision_at_k(recommended_items, relevant_items, k=10):
    """
    Вычисляет Average Precision at K.
    
    Args:
        recommended_items: список рекомендованных item_id
        relevant_items: список релевантных item_id (положительно оценённых)
        k: количество рекомендаций для оценки
    
    Returns:
        AP@K значение (0.0 до 1.0)
    """
    relevant_items = set(relevant_items)
    if not relevant_items:
        return 0.0
    
    score = 0.0
    num_hits = 0.0
    
    # Ограничиваем k длиной рекомендаций
    k = min(k, len(recommended_items))
    
    for i in range(k):
        if recommended_items[i] in relevant_items:
            num_hits += 1.0
            # Precision at i+1 = количество релевантных / (i+1)
            precision = num_hits / (i + 1)
            score += precision
    
    if num_hits == 0:
        return 0.0
    
    # Normalize by количество релевантных items
    return score / min(len(relevant_items), k)


def map_at_k(recommendations_dict, relevant_dict, k=10):
    """
    Вычисляет Mean Average Precision@K для всех пользователей.
    
    Args:
        recommendations_dict: {user_id: [item_id, ...]}
        relevant_dict: {user_id: [relevant_item_id, ...]}
        k: количество рекомендаций
    
    Returns:
        mAP@K значение
    """
    ap_scores = []
    
    for user_id in recommendations_dict:
        if user_id in relevant_dict and len(relevant_dict[user_id]) > 0:
            recs = recommendations_dict[user_id][:k]
            rels = relevant_dict[user_id]
            ap = average_precision_at_k(recs, rels, k)
            ap_scores.append(ap)
    
    if len(ap_scores) == 0:
        return 0.0
    
    return np.mean(ap_scores)

print("Функции AP@K и mAP@K определены.")


Функции AP@K и mAP@K определены.


In [22]:
# Шаг 2.2: Обучение ALS с базовыми параметрами
# Создаём модель ALS с базовыми параметрами из Implicit
model = implicit.als.AlternatingLeastSquares(
    factors=64,           # Размерность латентных факторов
    iterations=30,        # Количество итераций
    calculate_training_loss=True,
    random_state=42
)

# Обучаем модель на user-item матрице (НЕ транспонированной!)
model.fit(train_matrix)

print("Модель ALS обучена с базовыми параметрами:")
print(f"  factors=64, iterations=30")


100%|██████████| 30/30 [00:05<00:00,  5.92it/s, loss=0.00529]

Модель ALS обучена с базовыми параметрами:
  factors=64, iterations=30


In [23]:
# Шаг 2.3: Подготовка тестовых пользователей для оценки
import random

# Выбираем N случайных пользователей для тестирования
N_TEST_USERS = 500
random.seed(42)
sample_users = random.sample(list(user_index), N_TEST_USERS)

print(f"Выбрано {N_TEST_USERS} случайных пользователей для тестирования")

# Создаём словарь релевантных книг для каждого пользователя из теста
def get_relevant_books_for_user(user_id, train_df, test_df):
    """
    Получает релевантные книги для пользователя:
    - книги, которые пользователь оценил на 4 или 5 в тестовой выборке
    - книги, которые пользователь НЕ читал в обучающей выборке
    """
    train_books = set(train_df[train_df['user_id'] == user_id]['book_id'].unique())
    relevant = test_df[
        (test_df['user_id'] == user_id) & 
        (test_df['rating'] >= 4) &
        (~test_df['book_id'].isin(train_books))
    ]['book_id'].unique()
    return list(relevant)

# Создаём словарь релевантных книг
relevant_books_dict = {}
for user_id in sample_users:
    relevant_books_dict[user_id] = get_relevant_books_for_user(user_id, train_ratings, test_ratings)

# Фильтруем только пользователей с хотя бы одной релевантной книгой
users_with_relevants = [u for u in sample_users if len(relevant_books_dict[u]) > 0]
print(f"Пользователей с релевантными книгами в тесте: {len(users_with_relevants)}")

Выбрано 500 случайных пользователей для тестирования
Пользователей с релевантными книгами в тесте: 499


In [24]:
# Шаг 2.4: Генерация рекомендаций и расчёт mAP@10 для ALS
def get_als_recommendations(user_id, model, train_matrix, user_id_to_idx, book_id_to_idx, idx_to_book_id, N=30):
    """
    Получает рекомендации ALS для пользователя.
    """
    if user_id not in user_id_to_idx:
        return []
    
    user_idx = user_id_to_idx[user_id]
    
    # Получаем вектор взаимодействий пользователя из train матрицы
    user_items = train_matrix[user_idx].toarray().reshape(-1)
    
    # Получаем рекомендации
    item_ids, scores = model.recommend(
        user_idx,
        sparse.csr_matrix(user_items),
        N=N,
        filter_already_liked_items=True
    )
    
    # Преобразуем внутренние индексы в book_id
    recommended_book_ids = [idx_to_book_id[item_id] for item_id in item_ids]
    
    return recommended_book_ids

# Генерируем рекомендации для всех тестовых пользователей
als_recommendations = {}
for user_id in users_with_relevants:
    als_recommendations[user_id] = get_als_recommendations(
        user_id, model, train_matrix, 
        user_id_to_idx, book_id_to_idx, idx_to_book_id, N=30
    )

# Рассчитываем mAP@10 для ALS
als_map = map_at_k(als_recommendations, relevant_books_dict, k=10)

print(f"\nmAP@10 для ALS с базовыми параметрами: {als_map:.4f}")



mAP@10 для ALS с базовыми параметрами: 0.0006


In [25]:
# Шаг 2.5: Расчёт бейзлайнов для сравнения
# Бейзлайн 1: Случайные рекомендации
def get_random_recommendations(user_id, book_id_to_idx, k=10):
    """Случайные рекомендации из доступных книг."""
    all_books = list(book_id_to_idx.keys())
    return random.sample(all_books, min(k, len(all_books)))

random_recommendations = {}
for user_id in users_with_relevants:
    random_recommendations[user_id] = get_random_recommendations(
        user_id, book_id_to_idx, k=10
    )

random_baseline_map = map_at_k(random_recommendations, relevant_books_dict, k=10)
print(f"mAP@10 для случайного бейзлайна: {random_baseline_map:.4f}")

# Бейзлайн 2: Популярные книги (только на train данных!)
book_popularity = train_ratings.groupby('book_id').size().reset_index(name='count')
popular_books_sorted = book_popularity.sort_values('count', ascending=False)['book_id'].tolist()

def get_popular_recommendations(user_id, popular_books, train_df, k=10):
    """Рекомендует самые популярные книги, которые пользователь не читал."""
    train_books = set(train_df[train_df['user_id'] == user_id]['book_id'].unique())
    popular_unread = [b for b in popular_books if b not in train_books]
    return popular_unread[:k]

popular_recommendations = {}
for user_id in users_with_relevants:
    popular_recommendations[user_id] = get_popular_recommendations(
        user_id, popular_books_sorted, train_ratings, k=10
    )

popular_baseline_map = map_at_k(popular_recommendations, relevant_books_dict, k=10)
print(f"mAP@10 для бейзлайна популярных книг: {popular_baseline_map:.4f}")

print(f"\n=== Сравнение бейзлайнов ===")
print(f"Случайный бейзлайн:     mAP@10 = {random_baseline_map:.4f}")
print(f"Популярные книги:       mAP@10 = {popular_baseline_map:.4f}")
print(f"ALS (базовый):          mAP@10 = {als_map:.4f}")

mAP@10 для случайного бейзлайна: 0.0005
mAP@10 для бейзлайна популярных книг: 0.0000

=== Сравнение бейзлайнов ===
Случайный бейзлайн:     mAP@10 = 0.0005
Популярные книги:       mAP@10 = 0.0000
ALS (базовый):          mAP@10 = 0.0006


### Задание 3: Применение комбинации методов, подсчёт метрик

In [26]:
# Шаг 3.1: Подготовка контентных признаков - данные о книгах
# Загружаем данные о книгах для извлечения контентных признаков
books = pd.read_csv('books.csv')

# Выбираем релевантные колонки
books_features = books[['book_id', 'title', 'authors', 'average_rating', 'ratings_count']].copy()
books_features = books_features.dropna()
books_features['book_id'] = books_features['book_id'].astype(int)

print(f"Книг с признаками: {len(books_features)}")
print(books_features.head())

Книг с признаками: 10000
   book_id                                              title  \
0        1            The Hunger Games (The Hunger Games, #1)   
1        2  Harry Potter and the Sorcerer's Stone (Harry P...   
2        3                            Twilight (Twilight, #1)   
3        4                              To Kill a Mockingbird   
4        5                                   The Great Gatsby   

                       authors  average_rating  ratings_count  
0              Suzanne Collins            4.34        4780653  
1  J.K. Rowling, Mary GrandPré            4.44        4602479  
2              Stephenie Meyer            3.57        3866839  
3                   Harper Lee            4.25        3198671  
4          F. Scott Fitzgerald            3.89        2683664  


In [27]:
# Шаг 3.2: Векторизация заголовков книг с помощью Word2Vec
import nltk
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
import os

# Скачиваем необходимые ресурсы NLTK
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Токенизируем заголовки книг
def tokenize_text(text):
    """Токенизация и очистка текста."""
    if pd.isna(text):
        return []
    tokens = word_tokenize(str(text).lower())
    # Оставляем только буквенные токены
    tokens = [t for t in tokens if t.isalpha()]
    return tokens

# Создаём корпус заголовков
all_titles = books_features['title'].tolist()
tokenized_titles = [tokenize_text(title) for title in all_titles]

# Обучаем Word2Vec на заголовках книг
word2vec_model = Word2Vec(
    sentences=tokenized_titles, 
    vector_size=100,    # Размерность вектора
    window=5,           # Контекстное окно
    min_count=1,        # Минимальная частота слова
    workers=4,         # Параллельные потоки
    epochs=50,
    seed=42
)

print(f"Word2Vec модель обучена:")
print(f"  vocabulary size: {len(word2vec_model.wv)}")
print(f"  vector_size: {word2vec_model.wv.vector_size}")

# Функция для получения вектора заголовка книги
def get_title_vector(title, model):
    """Получает усреднённый вектор для заголовка книги."""
    tokens = tokenize_text(title)
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.wv.vector_size)

# Создаём векторы для всех книг
books_features['title_vector'] = books_features['title'].apply(
    lambda x: get_title_vector(x, word2vec_model)
)

# Сохраняем векторы
title_vectors = np.array(books_features['title_vector'].tolist())
np.save('title_vectors.npy', title_vectors)
print(f"Векторы заголовков сохранены: {title_vectors.shape}")

Word2Vec модель обучена:
  vocabulary size: 8981
  vector_size: 100
Векторы заголовков сохранены: (10000, 100)


In [28]:
# Шаг 3.3: Сбор признаков для пользователей и книг
from sklearn.preprocessing import StandardScaler

# Признаки книг из train данных
train_book_features = train_ratings.groupby('book_id').agg({
    'rating': ['mean', 'count']
}).reset_index()
train_book_features.columns = ['book_id', 'book_avg_rating', 'book_rating_count']

# Признаки пользователей из train данных
train_user_features = train_ratings.groupby('user_id').agg({
    'rating': ['mean', 'count', 'sum']
}).reset_index()
train_user_features.columns = ['user_id', 'user_avg_rating', 'user_book_count', 'user_total_liked']

# Объединяем с информацией о книгах
books_with_vectors = books_features[['book_id', 'title_vector']].copy()
train_book_features = train_book_features.merge(books_with_vectors, on='book_id', how='left')

print(f"Признаки книг: {train_book_features.shape}")
print(f"Признаки пользователей: {train_user_features.shape}")

Признаки книг: (6654, 4)
Признаки пользователей: (53424, 4)


In [30]:
# Шаг 3.4: Формирование обучающей выборки для классификатора
# Создаём пары (user, book) из train данных с положительными оценками (4-5)
positive_pairs = train_ratings[train_ratings['rating'] >= 4][['user_id', 'book_id']].copy()
negative_pairs = train_ratings[train_ratings['rating'] < 4][['user_id', 'book_id']].copy()

print(f"Позитивных пар: {len(positive_pairs)}")
print(f"Негативных пар: {len(negative_pairs)}")

# Добавляем баланс: берём столько же негативных пар, сколько позитивных
# Но если негативных пар меньше, чем позитивных, используем все доступные негативные пары
if len(negative_pairs) >= len(positive_pairs):
    # Достаточно негативных пар - выбираем случайные без повторений
    negative_sampled = negative_pairs.sample(n=len(positive_pairs), random_state=42)
else:
    # Недостаточно негативных пар - используем все доступные
    print(f"Внимание: недостаточно негативных примеров ({len(negative_pairs)}) для балансировки с позитивными ({len(positive_pairs)})")
    negative_sampled = negative_pairs.copy()

# Объединяем
training_pairs = pd.concat([
    positive_pairs.assign(label=1),
    negative_sampled.assign(label=0)
], ignore_index=True).sample(frac=1, random_state=42)

print(f"Всего пар для обучения: {len(training_pairs)}")
print(f"Позитивных меток: {sum(training_pairs['label'])}")
print(f"Негативных меток: {len(training_pairs) - sum(training_pairs['label'])}")

# Добавляем признаки к парам
def get_features_for_pair(user_id, book_id, user_features_df, book_features_df):
    """Извлекает признаки для пары user-book."""
    user_row = user_features_df[user_features_df['user_id'] == user_id]
    book_row = book_features_df[book_features_df['book_id'] == book_id]
    
    if user_row.empty or book_row.empty:
        return None
    
    features = {
        'user_avg_rating': user_row['user_avg_rating'].values[0],
        'user_book_count': user_row['user_book_count'].values[0],
        'user_total_liked': user_row['user_total_liked'].values[0],
        'book_avg_rating': book_row['book_avg_rating'].values[0],
        'book_rating_count': book_row['book_rating_count'].values[0],
    }
    return features

# Создаём DataFrame с признаками
features_list = []
for _, row in training_pairs.iterrows():
    feats = get_features_for_pair(
        row['user_id'], row['book_id'], 
        train_user_features, train_book_features
    )
    if feats:
        feats['user_id'] = row['user_id']
        feats['book_id'] = row['book_id']
        feats['label'] = row['label']
        features_list.append(feats)

features_df = pd.DataFrame(features_list)
print(f"\nСоздано признаковых записей: {len(features_df)}")
print(features_df.head())


Позитивных пар: 2903184
Негативных пар: 1256369
Внимание: недостаточно негативных примеров (1256369) для балансировки с позитивными (2903184)
Всего пар для обучения: 4159553
Позитивных меток: 2903184
Негативных меток: 1256369

Создано признаковых записей: 4159553
   user_avg_rating  user_book_count  user_total_liked  book_avg_rating  \
0         3.824324               74               283         4.088000   
1         4.232558               86               364         4.332047   
2         3.656566               99               362         4.100886   
3         3.075758               66               203         3.618644   
4         3.772727               88               332         3.970508   

   book_rating_count  user_id  book_id  label  
0                625    44000     1328      1  
1               6478     8182      155      1  
2               7563     9572       98      1  
3               1416     9810      879      0  
4               1458    13987      982      1  


In [32]:
# Шаг 3.5: Обучение классификатора (Random Forest)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Подготовка данных для классификатора
feature_cols = ['user_avg_rating', 'user_book_count', 'user_total_liked', 
                'book_avg_rating', 'book_rating_count']

X = features_df[feature_cols].values
y = features_df['label'].values

# Разделение на train/validation
X_train_cls, X_val_cls, y_train_cls, y_val_cls = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Обучение Random Forest
classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=8
)
classifier.fit(X_train_cls, y_train_cls)

# Оценка на validation
train_score = classifier.score(X_train_cls, y_train_cls)
val_score = classifier.score(X_val_cls, y_val_cls)

print(f"Точность на train: {train_score:.4f}")
print(f"Точность на validation: {val_score:.4f}")

# Важность признаков
importances = classifier.feature_importances_
print("\nВажность признаков:")
for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True):
    print(f"  {feat}: {imp:.4f}")

Точность на train: 0.7446
Точность на validation: 0.7441

Важность признаков:
  user_avg_rating: 0.6514
  book_avg_rating: 0.2543
  user_total_liked: 0.0482
  user_book_count: 0.0343
  book_rating_count: 0.0118


In [33]:
# Шаг 3.6: Гибридные рекомендации - ранжирование кандидатов ALS
from sklearn.preprocessing import MinMaxScaler

def get_hybrid_recommendations(user_id, als_recommendations, classifier, 
                                user_features_df, book_features_df,
                                user_id_to_idx, train_matrix, model,
                                idx_to_book_id, k=10):
    """
    Гибридные рекомендации:
    1. Получаем кандидатов от ALS
    2. Ранжируем с помощью классификатора
    """
    # Кандидаты от ALS
    candidates = als_recommendations.get(user_id, [])
    
    if len(candidates) == 0:
        return []
    
    # Формируем признаки для каждого кандидата
    candidate_features = []
    valid_candidates = []
    
    for book_id in candidates:
        feats = get_features_for_pair(user_id, book_id, user_features_df, book_features_df)
        if feats:
            candidate_features.append([
                feats['user_avg_rating'], feats['user_book_count'], feats['user_total_liked'],
                feats['book_avg_rating'], feats['book_rating_count']
            ])
            valid_candidates.append(book_id)
    
    if len(candidate_features) == 0:
        return candidates[:k]
    
    # Предсказания классификатора (вероятности)
    probs = classifier.predict_proba(np.array(candidate_features))[:, 1]
    
    # Ранжируем по вероятности (убывание)
    ranked_indices = np.argsort(-probs)
    ranked_candidates = [valid_candidates[i] for i in ranked_indices]
    
    return ranked_candidates[:k]

# Генерируем гибридные рекомендации для всех пользователей
hybrid_recommendations = {}
for user_id in users_with_relevants:
    hybrid_recommendations[user_id] = get_hybrid_recommendations(
        user_id, als_recommendations, classifier,
        train_user_features, train_book_features,
        user_id_to_idx, train_matrix, model,
        idx_to_book_id, k=10
    )

print(f"Гибридные рекомендации сгенерированы для {len(hybrid_recommendations)} пользователей")

Гибридные рекомендации сгенерированы для 499 пользователей


In [34]:
# Шаг 3.7: Расчёт mAP@10 для гибридных рекомендаций
# Рассчитываем mAP@10 для гибридных рекомендаций
hybrid_map = map_at_k(hybrid_recommendations, relevant_books_dict, k=10)

print(f"\n=== Сравнение результатов ===")
print(f"Случайный бейзлайн:     mAP@10 = {random_baseline_map:.4f}")
print(f"Популярные книги:       mAP@10 = {popular_baseline_map:.4f}")
print(f"ALS (базовый):          mAP@10 = {als_map:.4f}")
print(f"Гибридная модель:       mAP@10 = {hybrid_map:.4f}")

# Проверяем, побит ли бейзлайн ALS
if hybrid_map > als_map:
    print(f"\n✓ Гибридная модель побила ALS на {(hybrid_map - als_map)*100:.2f}%")
else:
    print(f"\n✗ Гибридная модель НЕ побила ALS на {(als_map - hybrid_map)*100:.2f}%")


=== Сравнение результатов ===
Случайный бейзлайн:     mAP@10 = 0.0005
Популярные книги:       mAP@10 = 0.0000
ALS (базовый):          mAP@10 = 0.0006
Гибридная модель:       mAP@10 = 0.0022

✓ Гибридная модель побила ALS на 0.16%


In [35]:
# Шаг 3.8: Анализ времени работы и выводы
import time

# Замеряем время генерации рекомендаций
start_time = time.time()
for user_id in users_with_relevants[:100]:
    _ = get_als_recommendations(user_id, model, train_matrix, 
                                 user_id_to_idx, book_id_to_idx, idx_to_book_id, N=30)
als_time = time.time() - start_time

start_time = time.time()
for user_id in users_with_relevants[:100]:
    _ = get_hybrid_recommendations(user_id, als_recommendations, classifier,
                                    train_user_features, train_book_features,
                                    user_id_to_idx, train_matrix, model,
                                    idx_to_book_id, k=10)
hybrid_time = time.time() - start_time

print(f"\n=== Анализ времени работы ===")
print(f"ALS (30 рекомендаций):      {als_time:.2f} сек для 100 пользователей")
print(f"Гибридная модель (топ-10):  {hybrid_time:.2f} сек для 100 пользователей")
print(f"Гибридная модель медленнее ALS в {hybrid_time/als_time:.2f} раз")

print(f"\n=== Итоговые выводы ===")
print(f"1. ALS (базовый):          mAP@10 = {als_map:.4f}")
print(f"2. Гибридная модель:       mAP@10 = {hybrid_map:.4f}")
print(f"3. Улучшение:              {(hybrid_map - als_map)*100:.2f}%")

if hybrid_map > als_map:
    print(f"\n✓ Гибридный подход (ALS + классификатор) показал лучшие результаты.")
    print(f"  Контентные признаки помогли улучшить качество рекомендаций.")
else:
    print(f"\n✗ Чистый ALS показал лучшие результаты.")
    print(f"  Контентные признаки не дали улучшения для данного датасета.")

print(f"\nВремя работы:")
print(f"  ALS быстрее гибридной модели в {hybrid_time/als_time:.1f} раз")
print(f"  ALS лучше для быстрых рекомендаций")
print(f"  Гибридная модель лучше для качественных рекомендаций (если улучшает mAP)")


=== Анализ времени работы ===
ALS (30 рекомендаций):      0.02 сек для 100 пользователей
Гибридная модель (топ-10):  3.17 сек для 100 пользователей
Гибридная модель медленнее ALS в 157.16 раз

=== Итоговые выводы ===
1. ALS (базовый):          mAP@10 = 0.0006
2. Гибридная модель:       mAP@10 = 0.0022
3. Улучшение:              0.16%

✓ Гибридный подход (ALS + классификатор) показал лучшие результаты.
  Контентные признаки помогли улучшить качество рекомендаций.

Время работы:
  ALS быстрее гибридной модели в 157.2 раз
  ALS лучше для быстрых рекомендаций
  Гибридная модель лучше для качественных рекомендаций (если улучшает mAP)
